In [ ]:
import numpy as np
import matplotlib.pyplot as plt
#import pydicom as dicom
import sys
import matplotlib.path as mplPath
import pandas as pd
import shutil
import os
import glob
import csv
from skimage import io
from tqdm import tqdm, trange

import torch
from torchvision import datasets
import torch.nn as nn
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset
from torchvision.io import read_image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader,WeightedRandomSampler

In [ ]:
from monai.transforms import LoadImage, Randomizable, apply_transform, Transpose, Rotate90, RandRotate
from monai.transforms import AddChannel, Compose, RandRotate90, Resize, ScaleIntensity, ToTensor, RandAffine, Rotate
from monai.utils import get_seed

depth, height, width = 40, 128, 128
rotation_angle = 90

train_transform = Compose([
    LoadImage(image_only=True),
    Transpose((2, 0, 1)),
    AddChannel(),
    ScaleIntensity(), 
    Resize((None, height, width)), 
    RandAffine( 
        prob=0.5,
        translate_range=(1, 5, 5),
        #rotate_range=(0, np.pi * 2, np.pi * 2),
        scale_range=(0.15, 0.15, 0.15),
        padding_mode='border'
    ),
    Rotate90(spatial_axes = (1, 2)),
    RandRotate(range_x = 45, prob = 0.5, keep_size = True),
    ToTensor()])

val_transform = Compose([LoadImage(image_only=True), Transpose((2, 0, 1)), AddChannel(), ScaleIntensity(), 
                         Resize((None, height, width)), Rotate90(spatial_axes = (1, 2)), ToTensor()])
test_transform = Compose([LoadImage(image_only=True), Transpose((2, 0, 1)), AddChannel(), ScaleIntensity(), 
                          Resize((None, height, width)), Rotate90(spatial_axes = (1, 2)), ToTensor()])

In [ ]:
import torchvision.transforms as transforms
from torchvision import datasets
import os

# train_transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(30),
#     #transforms.RandomVerticalFlip(),
#     transforms.Resize((64, 64)),
#     transforms.ToTensor()
# ])

# val_transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.Resize((64, 64)),
#     transforms.ToTensor()
# ])

# test_transform = transforms.Compose([
#     transforms.ToPILImage(),
#     transforms.Resize((64, 64)),
#     transforms.ToTensor()
# ])

def contrast_stretch(image, new_min=0, new_max=1):
    # Compute the current min and max intensity values
    current_min = np.min(image)
    current_max = np.max(image)
    
    # Apply contrast stretching
    stretched_image = (image - current_min) * (new_max - new_min) / (current_max - current_min) + new_min
    
    return stretched_image

def z_threshold(image):
    mean = np.mean(image)
    std = np.std(image)
    
    image_copy = image.copy()
    z_norm = (image - mean) / std
    
    for i in range(len(z_norm)):
        for j in range(len(z_norm[i])):
            if (z_norm[i][j] >= 4):
                image_copy[i][j] = 0
    return image_copy

def load_and_preprocess_images(file_path, transform):
    image_data = nib.load(file_path).get_fdata().transpose(2, 0, 1)
    
#     preprocessed_data = np.array([contrast_stretch(np.rot90(np.expand_dims(img, axis = -1), 
#                                            k = 1)) for img in image_data])[10:-10]
    
#     augmented_data = []
#     for i in preprocessed_data:
#         transformed = transform((i * 255).astype(np.uint8))
#         print(np.shape(transformed))
#         #print(np.shape(i))
#         augmented_data.append(transformed)
#     augmented_data = torch.stack(augmented_data).permute(1, 0, 2, 3)
    
    # turn transforms to monai transforms
    augmented_data = transform(file_path)[:, 10:-10]
    print(augmented_data.shape)
    #augmented_data = np.array(augmented_data)
    #print(np.shape(augmented_data))
    #print(s[:-7])
    return augmented_data
    #return augmented_data

import nibabel as nib

directory = 'Splitted'

x_train = []
y_train = []
x_val = []
y_val = []
x_test = []
y_test = []

used_patients_train = []
used_patients_val = []
used_patients_test = []


for subset in os.listdir(directory):
    if (subset != ".DS_Store"):
        print(subset)
        subset_path = os.path.join(directory, subset)
        for diagnosis in os.listdir(subset_path):
            if (diagnosis != ".DS_Store"):
                diagnosis_path = os.path.join(subset_path, diagnosis)
                for filename in os.listdir(diagnosis_path):
                    print(diagnosis)
                    if (filename != ".DS_Store"):
                        file_path = os.path.join(diagnosis_path, filename)

                        if (subset == 'train'):
                            x_train.append(load_and_preprocess_images(file_path, train_transform))

                            if (diagnosis == 'CN'):
                                y_train.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_train.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_train.append(2)
                            else:
                                y_train.append(3)
                            
                            used_patients_train.append(filename)

                        elif (subset == 'val'):
                            x_val.append(load_and_preprocess_images(file_path, val_transform))

                            if (diagnosis == 'CN'):
                                y_val.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_val.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_val.append(2)
                            else:
                                y_val.append(3)
                            
                            used_patients_val.append(filename)
                        else:
                            x_test.append(load_and_preprocess_images(file_path, test_transform))

                            if (diagnosis == 'CN'):
                                y_test.append(0)
                            elif (diagnosis == 'EMCI'):
                                y_test.append(1)
                            elif(diagnosis == 'LMCI'):
                                y_test.append(2)
                            else:
                                y_test.append(3)
                            
                            used_patients_test.append(filename)

In [ ]:
import numpy as np
def get_weighted_random_sample(label_list):
    class_frequencies = np.bincount(label_list)
    print(class_frequencies)
    # Calculate the inverse of class frequencies as probabilities
    class_probabilities = 1.0 / class_frequencies
    #print(class_probabilities)
    # Normalize the probabilities to sum to 1
    class_probabilities /= np.sum(class_probabilities)
    #print(self.label_list)
    return class_probabilities[label_list]
train_weights = get_weighted_random_sample(y_train)
val_weights = get_weighted_random_sample(y_val)
test_weights = get_weighted_random_sample(y_test)

In [ ]:
from torch.utils.data import WeightedRandomSampler
weighted_train_sampler = WeightedRandomSampler(
    weights=train_weights,
    num_samples=len(train_weights),
    replacement=True
)
weighted_val_sampler = WeightedRandomSampler(
    weights = val_weights,
    num_samples = len(val_weights),
    replacement = True
)
weighted_test_sampler = WeightedRandomSampler(
    weights = test_weights,
    num_samples = len(test_weights),
    replacement = True
)

In [ ]:
x_train, x_val, x_test = torch.stack(x_train), torch.stack(x_val), torch.stack(x_test)
y_train, y_val, y_test = torch.tensor(y_train), torch.tensor(y_val), torch.tensor(y_test)

In [ ]:
import matplotlib.pyplot as plt
plt.figure()
plt.imshow(x_train[4][:, 10].permute(1, 2, 0))
plt.show()
# ncdhw

In [ ]:
from torch.utils.data import TensorDataset
from torch.utils.data import DataLoader
batch_size = 32

train_set = TensorDataset(x_train, y_train)
val_set = TensorDataset(x_val, y_val)
test_set = TensorDataset(x_test, y_test)
train_loader = DataLoader(train_set, batch_size = batch_size, sampler = weighted_train_sampler)
val_loader = DataLoader(train_set, batch_size = batch_size, shuffle = True)
test_loader = DataLoader(test_set, batch_size = batch_size, shuffle = True)

In [ ]:
model = nn.Sequential(
    nn.Conv3d(1, 64, kernel_size = (3, 3, 3), stride = (1, 1, 1)),
    nn.ReLU(),
    nn.Conv3d(64, 64, kernel_size = (3, 3, 3), stride = (1, 1, 1)),
    nn.ReLU(),
    nn.Dropout(0.2, inplace = False),
    nn.MaxPool3d(kernel_size = (2, 3, 3), stride = (2, 2, 2)),
    nn.BatchNorm3d(64),
    nn.Conv3d(64, 128, kernel_size = (2, 3, 3), stride = (1, 1, 1)),
    nn.ReLU(),
    nn.Conv3d(128, 128, kernel_size = (2, 3, 3), stride = (1, 1, 1)),
    nn.ReLU(),
    nn.Dropout(0.2, inplace = False),
    nn.MaxPool3d(kernel_size = (2, 3, 3), stride = (1, 2, 2)),
    nn.BatchNorm3d(128),
    nn.Conv3d(128, 256, kernel_size = (1, 3, 3), stride = (1, 1, 1)),
    nn.ReLU(),
    nn.Conv3d(256, 256, kernel_size = (1, 3, 3), stride = (1, 1, 1)),
    nn.MaxPool3d(kernel_size = (1, 3, 3), stride = (1, 2, 2)),
    nn.BatchNorm3d(256),
    nn.AdaptiveAvgPool3d((1, 1, 1)),
    nn.Flatten(),
    nn.Linear(256, 512),
    nn.ReLU(),
    nn.Dropout(0.3, inplace = False),
    nn.Linear(512, 4),
    nn.Softmax(dim = -1)
)
model.train().cuda()

In [ ]:
from torchsummary import summary
summary(model, (1, 20, 128, 128))

In [ ]:
device = torch.device('cuda')
def train_loop(train_loader, val_loader, model, loss_fn, optimizer):
    size = len(train_loader.dataset)
    correct = 0
    samples = 0
    
    for batch, (X, y) in enumerate(train_loader):
        # Compute prediction and loss
        X, y = X.to(device), y.to(device)
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        optimizer.zero_grad()
#         for name, param in model.named_parameters():
#             if param.grad is not None:
#                 gradient_norm = torch.norm(param.grad)
#                print(gradient_norm)
        loss.backward()
        optimizer.step()
        
        with torch.no_grad():
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            samples += pred.size(0)
            if batch % 5 ==0:
                print(f"Acc: {float(correct) / float(samples) * 100:.2f}% Out of {samples} \n")
                correct = 0
                samples = 0

        if batch % 5 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]  \n")
    val_correct, val_samples = 0, 0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x)
        with torch.no_grad():
            val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            val_samples += pred.size(0)
    print(f"validation accuracy: {val_correct / val_samples * 100:.2f}%")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}% out of {size}, Avg loss: {test_loss:>8f} \n")

In [ ]:
criterion = nn.CrossEntropyLoss()
num_epochs = 15
optimizer = torch.optim.SGD(model.parameters(), lr= 0.001) 
for t in range(num_epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train_loop(train_loader, val_loader, model, criterion, optimizer)